# Ноутбук 02 — Стационарность, АКФ, ЧАКФ, АДФ-тест
**Подраздел 2.3 ПЗ — Стационарность и корреляционные свойства ряда**

Запускать после `01_preprocessing_features.ipynb`.

Артефакты:
- `reports/figures/fig_2_8_acf_original.png`  — Рисунок 2.8
- `reports/figures/fig_2_9_pacf_original.png` — Рисунок 2.9
- `reports/tables/table_2_2_adf_original.csv` — Таблица 2.2
- `data/interim/weekly_diff1.parquet`          — дифференцированный недельный ряд
- `reports/figures/fig_2_10_acf_diff1.png`    — Рисунок 2.10
- `reports/figures/fig_2_11_pacf_diff1.png`   — Рисунок 2.11
- `reports/tables/table_2_3_seasonal_acf.csv` — Подтверждение S = 52

## Ячейка 0 — Импорты и конфигурация

In [ ]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src.config import (
    DATA_INT, FIGURES, TABLES,
    SEASONAL_PERIOD,
)

FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# --- Константы анализа ---
# Глубина АКФ/ЧАКФ: 3 сезонных периода (156 недель), чтобы
# охватить пики на лагах 52, 104, 156 (п. 2.3.5)
LAGS            = 156
ALPHA           = 0.05       # уровень значимости для ДИ АКФ и АДФ-теста
# Спецификация АДФ-теста: константа + тренд (п. 2.3.3)
# Исключает ложное принятие H0 из-за детерминированных компонент
ADF_REGRESSION  = "ct"
SEASON          = SEASONAL_PERIOD          # 52 недели (из config.py)
SEASONAL_LAGS   = [SEASON, 2 * SEASON, 3 * SEASON]  # [52, 104, 156]

# --- Настройки оформления графиков ---
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

print(f"SEASONAL_PERIOD : {SEASON}")
print(f"LAGS            : {LAGS}")
print(f"ADF_REGRESSION  : {ADF_REGRESSION}")
print(f"SEASONAL_LAGS   : {SEASONAL_LAGS}")

## Ячейка 1 — Загрузка недельного ряда

Источник: `data/interim/weekly_sales.parquet` (артефакт Ячейки 7 ноутбука 01).

Индекс приводится к частоте `W-MON`.
Поскольку ряд формируется агрегацией дневных продаж, пропуски в индексе
возможны для нерабочих недель; они заполняются методом `pad` (forward-fill).
После заполнения ноутбук прерывается с ошибкой, если в ряде остались NaN.

In [ ]:
_raw = pd.read_parquet(DATA_INT / "weekly_sales.parquet")

# Столбец может называться 'sales' или быть единственным столбцом DataFrame
weekly = _raw.iloc[:, 0] if isinstance(_raw, pd.DataFrame) else _raw
weekly.index = pd.to_datetime(weekly.index)
weekly.name = "sales"

# Привод к явной частоте W-MON; пропуски заполняются pad (forward-fill)
weekly = weekly.asfreq("W-MON")
weekly = weekly.fillna(method="pad")

n_gaps = weekly.isna().sum()
assert n_gaps == 0, (
    f"После заполнения в weekly_sales остались пропуски: {n_gaps}. "
    "Запустите ноутбук 01 заново."
)

print(f"Наблюдений : {len(weekly)}")
print(f"Период     : {weekly.index[0].date()} — {weekly.index[-1].date()}")
print(f"Пропусков  : {n_gaps}  — OK")
print(f"Мин        : {weekly.min():,.0f}")
print(f"Макс       : {weekly.max():,.0f}")

## Вспомогательная функция построения АКФ / ЧАКФ

In [ ]:
def plot_correlation_function(
    series, lags, kind, title, fig_path,
    seasonal_lags=None, alpha=0.05
):
    """
    Строит и сохраняет АКФ или ЧАКФ.

    Parameters
    ----------
    series       : pd.Series — временной ряд
    lags         : int       — максимальная глубина лагов
    kind         : str       — 'acf' или 'pacf'
    title        : str       — заголовок графика
    fig_path     : Path      — путь для сохранения PNG
    seasonal_lags: list[int] — лаги, на которых ожидаются сезонные пики
    alpha        : float     — уровень значимости для ДИ Бартлетта
    """
    fig, ax = plt.subplots(figsize=(12, 4))

    if kind == "acf":
        plot_acf(series, lags=lags, alpha=alpha, ax=ax, zero=False)
    elif kind == "pacf":
        plot_pacf(series, lags=lags, alpha=alpha, ax=ax, zero=False, method="ywm")
    else:
        raise ValueError(f"kind должен быть 'acf' или 'pacf', получено: {kind!r}")

    ax.set_title(title)
    ax.set_xlabel("Лаг (недели)")
    ax.set_ylabel("Коэффициент корреляции")

    # Аннотация сезонных пиков (п. 2.3.1 / 2.3.5)
    if seasonal_lags:
        for lag in seasonal_lags:
            if lag <= lags:
                ax.axvline(x=lag, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
                ax.annotate(
                    f"lag={lag}",
                    xy=(lag, ax.get_ylim()[1] * 0.85),
                    fontsize=8, color="red", ha="center"
                )

    plt.tight_layout()
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Рисунок сохранён: {fig_path}")

print("Вспомогательная функция определена.")

## Ячейка 2 — АКФ исходного ряда (Рисунок 2.8)

**Ожидаемые результаты (п. 2.3.1):**
- Медленное, практически линейное затухание АКФ — признак трендовой нестационарности.
- Выраженные пики на лагах 52 и 104 — сезонная структура с периодом S = 52.

In [ ]:
plot_correlation_function(
    series=weekly,
    lags=LAGS,
    kind="acf",
    title="Рисунок 2.8 — АКФ исходного недельного ряда суммарных продаж",
    fig_path=FIGURES / "fig_2_8_acf_original.png",
    seasonal_lags=SEASONAL_LAGS,
    alpha=ALPHA,
)

## Ячейка 3 — ЧАКФ исходного ряда (Рисунок 2.9)

ЧАКФ дополняет картину АКФ и используется совместно с ней
для предварительной идентификации порядков p и q (п. 2.3.4).

In [ ]:
plot_correlation_function(
    series=weekly,
    lags=LAGS,
    kind="pacf",
    title="Рисунок 2.9 — ЧАКФ исходного недельного ряда суммарных продаж",
    fig_path=FIGURES / "fig_2_9_pacf_original.png",
    seasonal_lags=SEASONAL_LAGS,
    alpha=ALPHA,
)

## Ячейка 4 — АДФ-тест на исходном ряде (Таблица 2.2)

**Гипотезы (п. 2.3.2):**
- H0: единичный корень присутствует (ряд нестационарен)
- H1: ряд стационарен

**Спецификация (п. 2.3.3):** `regression='ct'` — константа + детерминированный тренд.
Поскольку структурные сдвиги 2014–2016 гг. зафиксированы в п. 2.2.3,
эта спецификация исключает ложное принятие H0 из-за детерминированных компонент.

In [ ]:
adf_result_orig = adfuller(weekly, regression=ADF_REGRESSION, autolag="AIC")

adf_stat_orig   = adf_result_orig[0]
adf_pvalue_orig = adf_result_orig[1]
adf_crit_orig   = adf_result_orig[4]   # словарь критических значений

# --- Сохранение Таблицы 2.2 ---
table_2_2 = pd.DataFrame({
    "Показатель": [
        "Тестовая статистика",
        "p-значение",
        "Крит. значение 1 %",
        "Крит. значение 5 %",
        "Крит. значение 10 %",
        "Вывод",
    ],
    "Значение": [
        f"{adf_stat_orig:.4f}",
        f"{adf_pvalue_orig:.4f}",
        f"{adf_crit_orig['1%']:.4f}",
        f"{adf_crit_orig['5%']:.4f}",
        f"{adf_crit_orig['10%']:.4f}",
        "H0 не отвергается" if adf_pvalue_orig > ALPHA else "H0 отвергается",
    ],
})

table_2_2.to_csv(TABLES / "table_2_2_adf_original.csv", index=False)
print(table_2_2.to_string(index=False))

# --- Формирование вывода (п. 2.3.2) ---
if adf_pvalue_orig > ALPHA:
    print(f"\np-значение ({adf_pvalue_orig:.4f}) > ALPHA ({ALPHA}) "
          "=> H0 не отвергается => ряд нестационарен.")
else:
    print(f"\np-значение ({adf_pvalue_orig:.4f}) <= ALPHA ({ALPHA}) "
          "=> H0 отвергается => ряд стационарен (d = 0).")

# Защита: для нестационарного ряда ожидаем p > ALPHA
# (если ряд окажется стационарным — уточнить п. 2.3.3)
assert adf_pvalue_orig > ALPHA, (
    "ВНИМАНИЕ: АДФ-тест отвергает H0 на исходном ряде. "
    "Пересмотрите спецификацию или проверьте данные (п. 2.3.3)."
)

## Ячейка 5 — Определение порядка d (п. 2.3.3)

Порядок обычного дифференцирования d задаётся как количественное следствие
результата АДФ-теста из Ячейки 4.

In [ ]:
if adf_pvalue_orig > ALPHA:
    D_ORDER = 1
    print(
        f"p-значение АДФ-теста ({adf_pvalue_orig:.4f}) > ALPHA ({ALPHA}): "
        f"H0 о единичном корне не отвергается => d = {D_ORDER}.\n"
        "Первое дифференцирование устранит детерминированный тренд "
        "и переведёт ряд в пространство приростов (Δy_t = y_t - y_{t-1})."
    )
else:
    D_ORDER = 0
    print(
        f"p-значение АДФ-теста ({adf_pvalue_orig:.4f}) <= ALPHA ({ALPHA}): "
        f"H0 отвергается => ряд стационарен => d = {D_ORDER}."
    )

print(f"\nD_ORDER = {D_ORDER}")

## Ячейка 6 — Первое дифференцирование (п. 2.3.4)

Применяется только если D_ORDER == 1.
Результирующий ряд сохраняется в `data/interim/weekly_diff1.parquet`.

In [ ]:
if D_ORDER == 1:
    diff1 = weekly.diff().dropna()
    diff1.name = "sales_diff1"
    diff1.to_frame().to_parquet(DATA_INT / "weekly_diff1.parquet")
    print(f"Дифференцированный ряд: {len(diff1)} наблюдений")
    print(f"Период: {diff1.index[0].date()} — {diff1.index[-1].date()}")
    print("Сохранён: data/interim/weekly_diff1.parquet")
else:
    diff1 = weekly.copy()
    diff1.name = "sales_diff0"
    print("D_ORDER = 0: дифференцирование не требуется; анализ проводится на исходном ряде.")

## Ячейка 7 — АКФ дифференцированного ряда (Рисунок 2.10)

**Ожидаемые результаты (п. 2.3.4):**
- Медленное затухание АКФ устранено => трендовая нестационарность устранена.
- Характер убывания АКФ используется для предварительной идентификации q.
- Остаточная структура в 2014–2015 и конце 2017 г. проверяется по значимым пикам
  за пределами ДИ Бартлетта.

In [ ]:
plot_correlation_function(
    series=diff1,
    lags=LAGS,
    kind="acf",
    title="Рисунок 2.10 — АКФ дифференцированного ряда (d = 1)",
    fig_path=FIGURES / "fig_2_10_acf_diff1.png",
    seasonal_lags=SEASONAL_LAGS,
    alpha=ALPHA,
)

## Ячейка 8 — ЧАКФ дифференцированного ряда (Рисунок 2.11)

Отсечение ЧАКФ используется для предварительной идентификации p несезонной части САРИМА (п. 2.3.4).

In [ ]:
plot_correlation_function(
    series=diff1,
    lags=LAGS,
    kind="pacf",
    title="Рисунок 2.11 — ЧАКФ дифференцированного ряда (d = 1)",
    fig_path=FIGURES / "fig_2_11_pacf_diff1.png",
    seasonal_lags=SEASONAL_LAGS,
    alpha=ALPHA,
)

## Ячейка 9 — АДФ-тест на дифференцированном ряде (валидация)

Подтверждает, что первое дифференцирование устранило единичный корень.
Ожидается: p <= ALPHA => H0 отвергается => diff1 стационарен.

In [ ]:
adf_result_diff = adfuller(diff1, regression=ADF_REGRESSION, autolag="AIC")

adf_stat_diff   = adf_result_diff[0]
adf_pvalue_diff = adf_result_diff[1]
adf_crit_diff   = adf_result_diff[4]

print(f"АДФ-статистика : {adf_stat_diff:.4f}")
print(f"p-значение      : {adf_pvalue_diff:.4f}")
print(f"Крит. 1 %       : {adf_crit_diff['1%']:.4f}")
print(f"Крит. 5 %       : {adf_crit_diff['5%']:.4f}")
print(f"Крит. 10 %      : {adf_crit_diff['10%']:.4f}")

if adf_pvalue_diff <= ALPHA:
    print(f"\np-значение ({adf_pvalue_diff:.4f}) <= ALPHA ({ALPHA}) "
          "=> H0 отвергается => дифференцированный ряд стационарен. d = 1 подтверждён.")
else:
    print(f"\nВНИМАНИЕ: p-значение ({adf_pvalue_diff:.4f}) > ALPHA ({ALPHA}). "
          "Рассмотрите d = 2 или повторную спецификацию.")

assert adf_pvalue_diff <= ALPHA, (
    f"Дифференцированный ряд всё ещё нестационарен (p={adf_pvalue_diff:.4f}). "
    "Проверьте данные или увеличьте d."
)

## Ячейка 10 — Статистическое подтверждение S = 52 (Таблица 2.3, п. 2.3.5)

Задача: не определить период заново, а статистически подтвердить
уже выявленную годовую сезонность как основание для выбора S = 52 в САРИМА.

Значимость пика на лаге k проверяется сравнением |ACF(k)| с
границей ДИ Бартлетта: 1.96 / sqrt(n) при уровне значимости 0.05.

In [ ]:
n = len(weekly)
acf_vals, acf_confint = acf(weekly, nlags=LAGS, alpha=ALPHA, fft=True)

# Граница ДИ Бартлетта (приближение для нестационарного ряда)
bartlett_bound = 1.96 / np.sqrt(n)

rows = []
for lag in SEASONAL_LAGS:
    val     = acf_vals[lag]
    ci_low  = acf_confint[lag, 0] - val   # смещение нижней границы
    ci_high = acf_confint[lag, 1] - val   # смещение верхней границы
    is_sig  = abs(val) > bartlett_bound
    rows.append({
        "Лаг (недели)": lag,
        "ACF": round(val, 4),
        "ДИ нижняя": round(val + ci_low, 4),
        "ДИ верхняя": round(val + ci_high, 4),
        "Граница Бартлетта": round(bartlett_bound, 4),
        "Значим (p=0.05)": "Да" if is_sig else "Нет",
    })

table_2_3 = pd.DataFrame(rows)
table_2_3.to_csv(TABLES / "table_2_3_seasonal_acf.csv", index=False)
print(table_2_3.to_string(index=False))

# Проверка: все три сезонных пика должны быть значимы
all_significant = all(abs(acf_vals[lag]) > bartlett_bound for lag in SEASONAL_LAGS)
if all_significant:
    print(f"\nВсе пики на лагах {SEASONAL_LAGS} значимы при уровне {ALPHA}. "
          f"S = {SEASON} статистически подтверждён.")
else:
    print(f"\nВНИМАНИЕ: не все сезонные пики значимы. "
          "Проверьте данные или пересмотрите выбор S.")

assert all_significant, (
    f"Сезонные пики на лагах {SEASONAL_LAGS} не подтверждены статистически. "
    f"Пересмотрите S или проверьте входные данные."
)